## Import libraries

In [2]:
from pathlib import Path

from torchvision.datasets import ImageFolder
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from transformations import train_transform, val_transform

from ultralytics import YOLO 

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(device)

cuda


## CNN


In [4]:
from torch import nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Flatten(),
            nn.LazyLinear(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.LazyLinear(5)
        )


    def forward(self, X):
        return self.network(X)

In [5]:
from ultralytics import YOLO

# Load a pre-trained YOLOv8 classification model
yolo_model = YOLO('yolov8n-cls.pt') 



## Training the model


In [6]:
model = CNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [7]:
path = Path("../VolleyballDataset")

if not path.exists():
   print("Dataset not found")
else:
    print("Volleyball Dataset Found")

BATCH = 32

torch.manual_seed(12345)

train_set = ImageFolder(path / "train", train_transform)
val_set = ImageFolder(path / "valid", val_transform)

# Data Loaders
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH)

Volleyball Dataset Found


In [8]:

def train(model, train_loader, val_loader, loss_fn, optimizer, epochs=10):
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        avg_train_loss = train_loss / len(train_loader)
        history["train_loss"].append(avg_train_loss)

        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_accuracy = 100 * correct / total
        history['val_loss'].append(val_accuracy)
        
        print(f"Epoch {epoch+1} Summary: Train Loss: {avg_train_loss:.4f} | Val Acc: {val_accuracy:.2f}%")
        
    return history

In [9]:
history = train(model, train_loader, val_loader, loss_fn, optimizer, epochs=10)

Epoch 1/10:   1%|▏         | 5/400 [00:01<02:31,  2.60batch/s, loss=1.54]


KeyboardInterrupt: 

In [10]:
# Train the model
results = yolo_model.train(
    data='../VolleyballDataset', # Path to your dataset root
    epochs=10,                   # Match your current CNN epochs
    imgsz=224,                   # Standard size for classification
    device=device                # Use the 'cuda' or 'mps' variable you defined
)

Ultralytics 8.4.37  Python-3.12.7 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3070, 8191MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../VolleyballDataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=

In [ ]:
torch.save(model.state_dict(), "weights/action_model.pt")